# Traffic Debug
Query `traffic_events.db` and review matching events with side-by-side clip playback.

In [ ]:
import os
import sqlite3
import subprocess
import tempfile
import pandas as pd
from pathlib import Path
from IPython.display import display, Video

PROJECT_ROOT = Path("/Users/jrill/Documents/traffic_project")
DB_PATH      = PROJECT_ROOT / "traffic_events.db"
DATA_DIR     = PROJECT_ROOT / 'traffic_data'

In [39]:
# ── Edit this query to filter events ────────────────────────
QUERY = """
SELECT *
FROM traffic_events
WHERE 
  1=1
  --AND direction = 'etw'
  --AND contains_parked = 1
  AND speed_mph > 30
ORDER BY RANDOM()
LIMIT 30
"""

In [ ]:
con = sqlite3.connect(DB_PATH)
df  = pd.read_sql(QUERY, con)
con.close()
display(df)

In [ ]:
con = sqlite3.connect(DB_PATH)
df  = pd.read_sql(QUERY, con)
con.close()
display(df)

In [ ]:
def _float(val):
    try:
        return float(val) if val is not None else 0.0
    except (ValueError, TypeError):
        return 0.0


def _clip_start_sec(clip_name):
    """Parse seconds-since-midnight from clip filename (HH.MM.SS-...)."""
    start_str = clip_name.split('[')[0].split('-')[0]
    h, m, s = start_str.split('.')
    return int(h) * 3600 + int(m) * 60 + int(s)


def stitch_side_by_side(wf_path, ef_path, wf_clip, ef_clip, height=480):
    """
    WF always left, EF always right.
    Clips are aligned by wall-clock start time: whichever clip starts later
    is padded by freezing its first frame until the other clip catches up.
    """
    wf_start = _clip_start_sec(wf_clip)
    ef_start = _clip_start_sec(ef_clip)
    wf_delay = max(0.0, wf_start - ef_start)
    ef_delay = max(0.0, ef_start - wf_start)

    d_wf = f'{wf_delay:.3f}'
    d_ef = f'{ef_delay:.3f}'
    lf = (f'[0:v]scale=-2:{height},setsar=1,'
          f'tpad=start_duration={d_wf}:start_mode=clone[l]'
          if wf_delay else f'[0:v]scale=-2:{height},setsar=1[l]')
    rf = (f'[1:v]scale=-2:{height},setsar=1,'
          f'tpad=start_duration={d_ef}:start_mode=clone[r]'
          if ef_delay else f'[1:v]scale=-2:{height},setsar=1[r]')

    tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
    tmp.close()
    subprocess.run([
        'ffmpeg', '-y',
        '-i', str(wf_path), '-i', str(ef_path),
        '-filter_complex', f'{lf};{rf};[l][r]hstack=inputs=2',
        '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
        tmp.name,
    ], capture_output=True, check=True)
    return tmp.name


def show_event(i, row):
    direction   = row['direction']
    motion_type = row.get('motion_type') or '?'
    parked_tag  = ' [parked]' if row.get('contains_parked') else ''
    wf_plate    = row.get('wf_plate')
    ef_plate    = row.get('ef_plate')
    wf_conf     = _float(row.get('wf_plate_conf'))
    ef_conf     = _float(row.get('ef_plate_conf'))
    plate       = row.get('plate')
    speed       = row.get('speed_mph')
    wf_clip     = row['wf_clip']
    ef_clip     = row['ef_clip']
    ds          = row['ds']

    wf_path = DATA_DIR / ds / 'WF' / 'mp4' / wf_clip
    ef_path = DATA_DIR / ds / 'EF' / 'mp4' / ef_clip

    print(f"Event {i+1}: {wf_clip}  +  {ef_clip}")
    print(f"  [{direction}] [{motion_type}]{parked_tag}  "
          f"WF: {wf_plate!r} ({wf_conf:.2f})  "
          f"EF: {ef_plate!r} ({ef_conf:.2f})  "
          f"-> {plate!r}  |  {speed} mph")

    tmp = stitch_side_by_side(wf_path, ef_path, wf_clip, ef_clip)
    display(Video(tmp, embed=True))
    os.unlink(tmp)


con  = sqlite3.connect(DB_PATH)
rows = pd.read_sql(QUERY, con).to_dict('records')
con.close()

print(f"{len(rows)} event(s) matched")
print('=' * 70)
for i, row in enumerate(rows):
    show_event(i, row)